In [ ]:
pip install pretty_midi

In [ ]:
!sudo apt install -y fluidsynth

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fluidsynth is already the newest version (2.2.5-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
pip install pyfluidsynth

In [ ]:
!pip install --upgrade pyfluidsynth

In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import collections
import pretty_midi
import glob
from IPython import display
import os
import fluidsynth

sampling_rate = 44100

In [ ]:
!mkdir music-midi-dataset    # create a directory to extract the dataset
!unzip /content/archive.zip  # replace the zip file name as per your file name

mkdir: cannot create directory ‘music-midi-dataset’: File exists
Archive:  /content/archive.zip
  inflating: midi_dataset/midi_dataset/x (1).mid  
  inflating: midi_dataset/midi_dataset/x (10).mid  
  inflating: midi_dataset/midi_dataset/x (11).mid  
  inflating: midi_dataset/midi_dataset/x (12).mid  
  inflating: midi_dataset/midi_dataset/x (13).mid  
  inflating: midi_dataset/midi_dataset/x (14).mid  
  inflating: midi_dataset/midi_dataset/x (15).mid  
  inflating: midi_dataset/midi_dataset/x (16).mid  
  inflating: midi_dataset/midi_dataset/x (17).mid  
  inflating: midi_dataset/midi_dataset/x (18).mid  
  inflating: midi_dataset/midi_dataset/x (19).mid  
  inflating: midi_dataset/midi_dataset/x (2).mid  
  inflating: midi_dataset/midi_dataset/x (20).mid  
  inflating: midi_dataset/midi_dataset/x (21).mid  
  inflating: midi_dataset/midi_dataset/x (22).mid  
  inflating: midi_dataset/midi_dataset/x (23).mid  
  inflating: midi_dataset/midi_dataset/x (24).mid  
  inflating: midi_data

Audio Playback

In [ ]:
def display_audio(pm, seconds=30):
    waveform = pm.fluidsynth(fs=sampling_rate)
    waveform_short = waveform[:seconds * sampling_rate]
    return display.Audio(waveform_short, rate=sampling_rate)

MIDI to Notes conversion

In [ ]:
def midi_to_notes(midi_file):
    pm = pretty_midi.PrettyMIDI(midi_file)
    # Check if there are any instruments in the MIDI file
    if not pm.instruments:
        print(f"Skipping {midi_file}: No instruments found.")
        return pd.DataFrame() # Return an empty DataFrame if no instruments

    instrument = pm.instruments[0]
    notes = collections.defaultdict(list)
    sorted_notes = sorted(instrument.notes, key=lambda note: note.start)
    if not sorted_notes: # Also check if the instrument has any notes
        print(f"Skipping {midi_file}: Instrument has no notes.")
        return pd.DataFrame()

    prev_start = sorted_notes[0].start

    for note in sorted_notes:
        start = note.start
        end = note.end
        notes["pitch"].append(note.pitch)
        notes["start"].append(start)
        notes["end"].append(end)
        notes["step"].append(start - prev_start)
        notes["duration"].append(end - start)
        prev_start = start

    return pd.DataFrame({name: np.array(value) for name, value in notes.items()})

Load MIDI Files

In [ ]:
filenames = glob.glob('*.mid')
if not filenames:
    raise FileNotFoundError("❌ No MIDI files found in the current directory.")

all_notes = []
for filename in filenames:
    raw_notes = midi_to_notes(filename)
    if not raw_notes.empty:
        all_notes.append(raw_notes)
    else:
        print(f"Skipping processing for {filename} as it returned an empty DataFrame.")



# Concatenate notes from all valid files
combined_notes = pd.concat(all_notes, ignore_index=True)

key_order = ["pitch", "step", "duration"]
sample_notes = np.stack([combined_notes[key] for key in key_order], axis=1)

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Create Sequence

In [ ]:
seq_length = 20
vocab_size = 128

def create_sequences(notes, seq_length):
    sequences = []
    targets = []
    for i in range(len(notes) - seq_length):
        sequence = notes[i:i + seq_length] / [vocab_size, 1, 1]
        target = notes[i + seq_length] / [vocab_size, 1, 1]
        sequences.append(sequence)
        targets.append(target)

    sequences = np.array(sequences)
    targets = np.array(targets)
    return tf.data.Dataset.from_tensor_slices(
        (sequences, {
            "pitch": tf.cast(targets[:, 0], tf.int32),
            "step": tf.expand_dims(targets[:, 1], -1),
            "duration": tf.expand_dims(targets[:, 2], -1)
        })
    )

train_notes = sample_notes
seq_ds = create_sequences(train_notes, seq_length)
train_ds = seq_ds.shuffle(1000).batch(64)

Build LSTM Model

In [ ]:
input_data = tf.keras.Input(shape=(seq_length, 3))
x = tf.keras.layers.LSTM(128)(input_data)
outputs = {
    "pitch": tf.keras.layers.Dense(64, name="pitch")(x),
    "step": tf.keras.layers.Dense(1, name="step")(x),
    "duration": tf.keras.layers.Dense(1, name="duration")(x),
}
model = tf.keras.Model(input_data, outputs)

model.compile(
    loss={
        "pitch": tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        "step": tf.keras.losses.MeanSquaredError(),
        "duration": tf.keras.losses.MeanSquaredError(),
    },
    loss_weights={"pitch": 0.05, "step": 1.0, "duration": 1.0},
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005)
)


Train model

In [ ]:
model.fit(train_ds, epochs=10)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - duration_loss: 0.0264 - loss: 0.3798 - pitch_loss: 4.0255 - step_loss: 0.1512
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - duration_loss: 0.0072 - loss: 0.2777 - pitch_loss: 3.7944 - step_loss: 0.0786
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - duration_loss: 0.0087 - loss: 0.2650 - pitch_loss: 3.2189 - step_loss: 0.0940
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - duration_loss: 0.0207 - loss: 0.1765 - pitch_loss: 1.3711 - step_loss: 0.0883
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - duration_loss: 0.0286 - loss: 0.1534 - pitch_loss: 0.0425 - step_loss: 0.1214
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - duration_loss: 0.0159 - loss: 0.1057 - pitch_loss: 0.0134 - step_loss: 0.0906
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - duration_loss: 0.0147 - loss: 0.1054 - pitch_loss: 0.0082 - step_loss: 0.0916
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - duration_loss: 0.0115 - loss: 0.0999 - pitch_l

Save model to local disk

In [ ]:
model_save_path = "saved_music_lstm_model.keras"
model.save(model_save_path)
print(f"Model saved to: {os.path.abspath(model_save_path)}")

Model saved to: /content/saved_music_lstm_model.keras


Prediciton Function

In [ ]:
def predict_next_note(notes, model, temperature=1.0):
    inputs = np.expand_dims(notes, 0)
    predictions = model.predict(inputs, verbose=0)
    pitch_logits = predictions["pitch"] / temperature
    step = predictions["step"]
    duration = predictions["duration"]

    pitch = tf.random.categorical(pitch_logits, num_samples=1)
    pitch = tf.squeeze(pitch, axis=-1)
    step = tf.maximum(0, tf.squeeze(step, axis=-1))
    duration = tf.maximum(0, tf.squeeze(duration, axis=-1))

    return int(pitch), float(step), float(duration)

Generate music with multiple instrument

In [ ]:
instrument_names = ["Acoustic Grand Piano", "Violin", "Electric Bass (finger)"]
generated_tracks = []

for idx, instrument_name in enumerate(instrument_names):
    input_notes = sample_notes[:seq_length] / [vocab_size, 1, 1]
    prev_start = 0
    generated_notes = []

    for _ in range(500):
        pitch, step, duration = predict_next_note(input_notes, model, temperature=1.5)
        start = prev_start + step
        end = start + duration
        input_note = (pitch, step, duration)
        generated_notes.append((*input_note, start, end))
        input_notes = np.delete(input_notes, 0, axis=0)
        input_notes = np.append(input_notes, [input_note], axis=0)
        prev_start = start

    notes_df = pd.DataFrame(generated_notes, columns=(*key_order, "start", "end"))
    instrument_program = pretty_midi.instrument_name_to_program(instrument_name)
    midi_instrument = pretty_midi.Instrument(program=instrument_program, name=instrument_name)

    for _, row in notes_df.iterrows():
        note = pretty_midi.Note(
            velocity=100,
            pitch=int(row["pitch"]),
            start=float(row["start"]),
            end=float(row["end"]),
        )
        midi_instrument.notes.append(note)

    generated_tracks.append(midi_instrument)

Combine and save final song


In [ ]:
final_song = pretty_midi.PrettyMIDI()
for track in generated_tracks:
    final_song.instruments.append(track)

final_song.write("multi_instrument_music.mid")
display_audio(final_song, seconds=60)